# ECE 685D Final Project: Hypernetworks for Implicit Neural Representations

This notebook consolidates all 5 experiments into a single runnable file for Google Colab.

**Contents:**
1. Coordinate Network Baseline (single image INR)
2. Hypernetwork Training on CIFAR-10
3. Downstream Classification
4. CelebA Architecture Comparison
5. Super-Resolution via INR

---
## Setup: Install Dependencies & Configure Environment

In [ ]:
!pip install -q lpips scikit-learn pandas

In [ ]:
import math
import random
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import datasets, transforms
from tqdm import tqdm

from sklearn.decomposition import PCA

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Create output directories
for d in ['outputs/figures/reconstructions', 'outputs/figures/comparisons',
          'outputs/figures/super_resolution', 'outputs/models/checkpoints',
          'outputs/logs']:
    Path(d).mkdir(parents=True, exist_ok=True)

---
## Source Code: All Module Definitions

The following cells define all model architectures, data loaders, training loops, metrics, and utilities that were originally in the `src/` package.

### Utilities: Seed, Coordinates, Logger

In [ ]:
# ============================================================
# src/utils/seed.py - Reproducibility utilities
# ============================================================

def set_seed(seed=42):
    """Set random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# src/utils/coords.py - Coordinate grid utilities
# ============================================================

def get_coordinate_grid(H, W, normalize=True):
    """
    Generate a 2D coordinate grid.
    Returns: coords: (H*W, 2) tensor of (x, y) coordinates
    """
    if normalize:
        y = torch.linspace(-1, 1, H)
        x = torch.linspace(-1, 1, W)
    else:
        y = torch.arange(H, dtype=torch.float32) / (H - 1)
        x = torch.arange(W, dtype=torch.float32) / (W - 1)
    grid_y, grid_x = torch.meshgrid(y, x, indexing='ij')
    coords = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=-1)
    return coords


def image_to_coord_target(image):
    """
    Convert an image to (coords, target_pixels) pair for INR training.
    Args: image: (C, H, W) tensor in [0, 1]
    Returns: coords: (H*W, 2), pixels: (H*W, C)
    """
    C, H, W = image.shape
    coords = get_coordinate_grid(H, W)
    pixels = image.permute(1, 2, 0).reshape(H * W, C)
    return coords, pixels


def reconstruct_image(model, H, W, device='cuda', batch_size=None):
    """
    Reconstruct an image from a trained coordinate network.
    Returns: image: (C, H, W) tensor in [0, 1]
    """
    coords = get_coordinate_grid(H, W).to(device)
    model.eval()
    with torch.no_grad():
        if batch_size is None:
            pixels = model(coords)
        else:
            chunks = []
            for i in range(0, coords.shape[0], batch_size):
                chunk = model(coords[i:i+batch_size])
                chunks.append(chunk)
            pixels = torch.cat(chunks, dim=0)
    pixels = pixels.clamp(0, 1)
    C = pixels.shape[-1]
    image = pixels.reshape(H, W, C).permute(2, 0, 1)
    return image


def reconstruct_from_hypernetwork(weights, biases, H, W, omega_0=30.0, device='cuda'):
    """
    Reconstruct images from hypernetwork-generated weights.
    Returns: images: (B, C, H, W)
    """
    coords = get_coordinate_grid(H, W).to(device)
    with torch.no_grad():
        pred = apply_generated_weights(coords, weights, biases, omega_0)
    B = pred.shape[0]
    C = pred.shape[-1]
    images = pred.clamp(0, 1).reshape(B, H, W, C).permute(0, 3, 1, 2)
    return images


# ============================================================
# src/utils/logger.py - Experiment logging
# ============================================================

class ExperimentLogger:
    """Simple experiment logger that saves metrics to JSON."""
    def __init__(self, log_dir='outputs/logs', experiment_name='experiment'):
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(parents=True, exist_ok=True)
        self.experiment_name = experiment_name
        self.metrics = {}
        self.start_time = time.time()

    def log(self, step, **kwargs):
        for key, value in kwargs.items():
            if key not in self.metrics:
                self.metrics[key] = []
            self.metrics[key].append({'step': step, 'value': value})

    def log_epoch(self, epoch, **kwargs):
        self.log(epoch, **kwargs)

    def save(self):
        output = {
            'experiment_name': self.experiment_name,
            'total_time': time.time() - self.start_time,
            'metrics': self.metrics,
        }
        path = self.log_dir / f'{self.experiment_name}.json'
        with open(path, 'w') as f:
            json.dump(output, f, indent=2)
        print(f'Saved experiment log to {path}')

    def summary(self):
        print(f'\n--- {self.experiment_name} Summary ---')
        print(f'Duration: {time.time() - self.start_time:.1f}s')
        for key, values in self.metrics.items():
            if values:
                last = values[-1]['value']
                print(f'  {key}: {last:.6f} (final)')

### Models: SIREN, Coordinate MLPs, Hypernetwork

In [ ]:
# ============================================================
# src/models/coordinate_mlp.py - Coordinate-based MLPs for INR
# ============================================================

class FourierFeatureMapping(nn.Module):
    """
    Random Fourier feature mapping: gamma(x) = [sin(Bx), cos(Bx)]
    Reference: Tancik et al., NeurIPS 2020
    """
    def __init__(self, in_features=2, mapping_size=256, scale=10.0):
        super().__init__()
        B = torch.randn(in_features, mapping_size) * scale
        self.register_buffer('B', B)
        self.out_features = mapping_size * 2

    def forward(self, x):
        x_proj = x @ self.B
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)


class ReLUCoordinateMLP(nn.Module):
    """Simple ReLU MLP for coordinate-based representations (spectral bias baseline)."""
    def __init__(self, in_features=2, hidden_features=256, hidden_layers=3, out_features=3):
        super().__init__()
        layers = []
        layers.append(nn.Linear(in_features, hidden_features))
        layers.append(nn.ReLU(inplace=True))
        for _ in range(hidden_layers):
            layers.append(nn.Linear(hidden_features, hidden_features))
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Linear(hidden_features, out_features))
        self.net = nn.Sequential(*layers)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, coords):
        return self.net(coords)


class FourierFeatureNetwork(nn.Module):
    """Fourier Feature Network = Fourier mapping + ReLU MLP."""
    def __init__(self, in_features=2, mapping_size=256, scale=10.0,
                 hidden_features=256, hidden_layers=3, out_features=3):
        super().__init__()
        self.fourier_mapping = FourierFeatureMapping(in_features, mapping_size, scale)
        self.mlp = ReLUCoordinateMLP(
            in_features=self.fourier_mapping.out_features,
            hidden_features=hidden_features,
            hidden_layers=hidden_layers,
            out_features=out_features,
        )

    def forward(self, coords):
        mapped = self.fourier_mapping(coords)
        return self.mlp(mapped)


# ============================================================
# src/models/siren.py - SIREN: Sinusoidal Representation Networks
# Reference: Sitzmann et al., NeurIPS 2020
# ============================================================

class SineLayer(nn.Module):
    """Linear layer followed by sine activation with special initialization."""
    def __init__(self, in_features, out_features, bias=True, is_first=False, omega_0=30.0):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.in_features = in_features
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self.init_weights()

    def init_weights(self):
        with torch.no_grad():
            if self.is_first:
                self.linear.weight.uniform_(-1 / self.in_features, 1 / self.in_features)
            else:
                self.linear.weight.uniform_(
                    -math.sqrt(6 / self.in_features) / self.omega_0,
                    math.sqrt(6 / self.in_features) / self.omega_0,
                )

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


class SIREN(nn.Module):
    """Full SIREN network for implicit neural representations."""
    def __init__(self, in_features=2, hidden_features=256, hidden_layers=3,
                 out_features=3, outermost_linear=True,
                 first_omega_0=30.0, hidden_omega_0=30.0):
        super().__init__()
        self.in_features = in_features
        self.hidden_features = hidden_features
        self.hidden_layers = hidden_layers
        self.out_features = out_features
        layers = []
        layers.append(SineLayer(in_features, hidden_features, is_first=True, omega_0=first_omega_0))
        for _ in range(hidden_layers):
            layers.append(SineLayer(hidden_features, hidden_features, is_first=False, omega_0=hidden_omega_0))
        if outermost_linear:
            final_linear = nn.Linear(hidden_features, out_features)
            with torch.no_grad():
                final_linear.weight.uniform_(
                    -math.sqrt(6 / hidden_features) / hidden_omega_0,
                    math.sqrt(6 / hidden_features) / hidden_omega_0,
                )
            layers.append(final_linear)
        else:
            layers.append(SineLayer(hidden_features, out_features, is_first=False, omega_0=hidden_omega_0))
        self.net = nn.Sequential(*layers)

    def forward(self, coords):
        return self.net(coords)

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())


# ============================================================
# src/models/hypernetwork.py - Hypernetwork: H_phi(I) -> theta
# ============================================================

class HyperNetwork(nn.Module):
    """
    Hypernetwork that takes an image and produces SIREN weights.
    Architecture: Encoder (CNN) -> Latent z -> Weight generators (per-layer MLPs)
    """
    def __init__(self, img_shape=(3, 32, 32), latent_dim=256,
                 siren_in_features=2, siren_hidden_features=256,
                 siren_hidden_layers=3, siren_out_features=3):
        super().__init__()
        self.latent_dim = latent_dim
        self.siren_in_features = siren_in_features
        self.siren_hidden_features = siren_hidden_features
        self.siren_hidden_layers = siren_hidden_layers
        self.siren_out_features = siren_out_features
        C, H, W = img_shape
        self.encoder = self._build_encoder(C, H, W, latent_dim)
        self.weight_generators = nn.ModuleList()
        self.bias_generators = nn.ModuleList()
        self.layer_shapes = self._compute_layer_shapes()
        for (fan_in, fan_out) in self.layer_shapes:
            num_weights = fan_in * fan_out
            num_biases = fan_out
            self.weight_generators.append(nn.Sequential(
                nn.Linear(latent_dim, latent_dim),
                nn.ReLU(inplace=True),
                nn.Linear(latent_dim, num_weights),
            ))
            self.bias_generators.append(nn.Sequential(
                nn.Linear(latent_dim, latent_dim // 2),
                nn.ReLU(inplace=True),
                nn.Linear(latent_dim // 2, num_biases),
            ))

    def _build_encoder(self, C, H, W, latent_dim):
        if H <= 32:
            encoder = nn.Sequential(
                nn.Conv2d(C, 64, 3, stride=2, padding=1),
                nn.BatchNorm2d(64), nn.ReLU(inplace=True),
                nn.Conv2d(64, 128, 3, stride=2, padding=1),
                nn.BatchNorm2d(128), nn.ReLU(inplace=True),
                nn.Conv2d(128, 256, 3, stride=2, padding=1),
                nn.BatchNorm2d(256), nn.ReLU(inplace=True),
                nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                nn.Linear(256, latent_dim), nn.ReLU(inplace=True),
            )
        else:
            encoder = nn.Sequential(
                nn.Conv2d(C, 64, 4, stride=2, padding=1),
                nn.BatchNorm2d(64), nn.ReLU(inplace=True),
                nn.Conv2d(64, 128, 4, stride=2, padding=1),
                nn.BatchNorm2d(128), nn.ReLU(inplace=True),
                nn.Conv2d(128, 256, 4, stride=2, padding=1),
                nn.BatchNorm2d(256), nn.ReLU(inplace=True),
                nn.Conv2d(256, 512, 4, stride=2, padding=1),
                nn.BatchNorm2d(512), nn.ReLU(inplace=True),
                nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                nn.Linear(512, latent_dim), nn.ReLU(inplace=True),
            )
        return encoder

    def _compute_layer_shapes(self):
        shapes = []
        shapes.append((self.siren_in_features, self.siren_hidden_features))
        for _ in range(self.siren_hidden_layers):
            shapes.append((self.siren_hidden_features, self.siren_hidden_features))
        shapes.append((self.siren_hidden_features, self.siren_out_features))
        return shapes

    def forward(self, images):
        latent = self.encoder(images)
        weights, biases = [], []
        for i, (fan_in, fan_out) in enumerate(self.layer_shapes):
            w = self.weight_generators[i](latent).view(-1, fan_out, fan_in)
            b = self.bias_generators[i](latent)
            weights.append(w)
            biases.append(b)
        return weights, biases, latent

    def get_latent(self, images):
        return self.encoder(images)


def apply_generated_weights(coords, weights, biases, omega_0=30.0):
    """Forward pass through a SIREN using generated weights."""
    if coords.dim() == 2:
        coords = coords.unsqueeze(0).expand(weights[0].shape[0], -1, -1)
    x = coords
    num_layers = len(weights)
    for i in range(num_layers):
        w = weights[i]
        b = biases[i]
        x = torch.bmm(x, w.transpose(1, 2)) + b.unsqueeze(1)
        if i < num_layers - 1:
            x = torch.sin(omega_0 * x)
    return x

### Data: CIFAR-10, CelebA, Transforms

In [ ]:
# ============================================================
# src/data/cifar.py - CIFAR-10 data loading
# ============================================================

class CIFAR10INRDataset(Dataset):
    """Wraps CIFAR-10 for INR training. Images normalized to [0, 1]."""
    def __init__(self, root='./data', train=True, download=True):
        self.dataset = datasets.CIFAR10(
            root=root, train=train, download=download,
            transform=transforms.ToTensor(),
        )
        self.classes = self.dataset.classes

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        return image, label


def get_cifar10_loaders(root='./data', batch_size=64, num_workers=2):
    """Get train and test DataLoaders for CIFAR-10."""
    train_dataset = CIFAR10INRDataset(root=root, train=True)
    test_dataset = CIFAR10INRDataset(root=root, train=False)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    return train_loader, test_loader


# ============================================================
# src/data/celeba.py - CelebA data loading
# ============================================================

class CelebAINRDataset(Dataset):
    """CelebA dataset for INR training. Crops and resizes faces."""
    def __init__(self, root='./data', split='train', image_size=64, download=False):
        transform = transforms.Compose([
            transforms.CenterCrop(178),
            transforms.Resize(image_size),
            transforms.ToTensor(),
        ])
        self.dataset = datasets.CelebA(
            root=root, split=split, download=download,
            transform=transform,
        )
        self.image_size = image_size

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, attr = self.dataset[idx]
        return image, attr


def get_celeba_loaders(root='./data', image_size=64, batch_size=32,
                       num_workers=2, max_train=None):
    """Get train/val/test DataLoaders for CelebA."""
    train_dataset = CelebAINRDataset(root=root, split='train', image_size=image_size)
    val_dataset = CelebAINRDataset(root=root, split='valid', image_size=image_size)
    test_dataset = CelebAINRDataset(root=root, split='test', image_size=image_size)
    if max_train is not None:
        train_dataset = torch.utils.data.Subset(
            train_dataset, range(min(max_train, len(train_dataset)))
        )
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    return train_loader, val_loader, test_loader


# ============================================================
# src/data/transforms.py - Image transforms
# ============================================================

def downsample_images(images, target_size):
    """Downsample a batch of images for super-resolution experiments."""
    if isinstance(target_size, int):
        target_size = (target_size, target_size)
    return F.interpolate(images, size=target_size, mode='bilinear', align_corners=False)

### Training: INR and Hypernetwork Loops

In [ ]:
# ============================================================
# src/training/train_inr.py - Single-image INR training
# ============================================================

def train_single_image_inr(model, coords, target_pixels, num_steps=2000,
                           lr=1e-4, device='cuda', log_interval=100):
    """Train a coordinate network on a single image."""
    model = model.to(device)
    coords = coords.to(device)
    target_pixels = target_pixels.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_steps)
    losses = []
    pbar = tqdm(range(num_steps), desc='Training INR')
    for step in pbar:
        optimizer.zero_grad()
        pred = model(coords)
        loss = F.mse_loss(pred, target_pixels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())
        if step % log_interval == 0:
            psnr = -10 * torch.log10(loss).item()
            pbar.set_postfix(loss=f'{loss.item():.6f}', psnr=f'{psnr:.2f}')
    return model, losses

### Evaluation: Metrics and Visualization

In [ ]:
# ============================================================
# src/evaluation/metrics.py - PSNR, SSIM, LPIPS
# ============================================================

def compute_psnr(pred, target, max_val=1.0):
    """Compute Peak Signal-to-Noise Ratio."""
    mse = F.mse_loss(pred, target, reduction='none')
    if pred.dim() == 4:
        mse = mse.mean(dim=(1, 2, 3))
    else:
        mse = mse.mean()
    psnr = 10 * torch.log10(max_val ** 2 / (mse + 1e-10))
    return psnr


def _gaussian_kernel(size=11, sigma=1.5, channels=3, device='cpu'):
    coords = torch.arange(size, dtype=torch.float32, device=device) - size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = torch.outer(g, g)
    g = g / g.sum()
    kernel = g.unsqueeze(0).unsqueeze(0).repeat(channels, 1, 1, 1)
    return kernel


def compute_ssim(pred, target, window_size=11, max_val=1.0):
    """Compute Structural Similarity Index Measure."""
    if pred.dim() == 3:
        pred = pred.unsqueeze(0)
        target = target.unsqueeze(0)
    C = pred.shape[1]
    kernel = _gaussian_kernel(window_size, sigma=1.5, channels=C, device=pred.device)
    mu1 = F.conv2d(pred, kernel, padding=window_size // 2, groups=C)
    mu2 = F.conv2d(target, kernel, padding=window_size // 2, groups=C)
    mu1_sq, mu2_sq, mu1_mu2 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    sigma1_sq = F.conv2d(pred * pred, kernel, padding=window_size // 2, groups=C) - mu1_sq
    sigma2_sq = F.conv2d(target * target, kernel, padding=window_size // 2, groups=C) - mu2_sq
    sigma12 = F.conv2d(pred * target, kernel, padding=window_size // 2, groups=C) - mu1_mu2
    C1 = (0.01 * max_val) ** 2
    C2 = (0.03 * max_val) ** 2
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_map.mean(dim=(1, 2, 3))


class LPIPSMetric:
    """LPIPS perceptual metric wrapper."""
    def __init__(self, net='alex', device='cuda'):
        try:
            import lpips
            self.loss_fn = lpips.LPIPS(net=net).to(device)
            self.loss_fn.eval()
            self.device = device
            self.available = True
        except ImportError:
            print("Warning: lpips package not installed. Run: pip install lpips")
            self.available = False

    @torch.no_grad()
    def compute(self, pred, target):
        if not self.available:
            return torch.zeros(pred.shape[0])
        pred_scaled = pred * 2 - 1
        target_scaled = target * 2 - 1
        return self.loss_fn(pred_scaled.to(self.device),
                           target_scaled.to(self.device)).squeeze()


def compute_all_metrics(pred, target, lpips_metric=None):
    """Compute PSNR, SSIM, and optionally LPIPS."""
    psnr = compute_psnr(pred, target).mean().item()
    ssim = compute_ssim(pred, target).mean().item()
    results = {'psnr': psnr, 'ssim': ssim}
    if lpips_metric is not None and lpips_metric.available:
        lpips_val = lpips_metric.compute(pred, target).mean().item()
        results['lpips'] = lpips_val
    return results

In [ ]:
# ============================================================
# src/evaluation/visualize.py - Plotting utilities
# ============================================================

def tensor_to_image(tensor):
    """Convert a (C, H, W) or (H, W, C) tensor to numpy image."""
    if isinstance(tensor, torch.Tensor):
        tensor = tensor.detach().cpu()
        if tensor.dim() == 3 and tensor.shape[0] in (1, 3):
            tensor = tensor.permute(1, 2, 0)
        tensor = tensor.numpy()
    return np.clip(tensor, 0, 1)


def plot_reconstruction(original, reconstructed, title=None, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(tensor_to_image(original))
    axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(tensor_to_image(reconstructed))
    axes[1].set_title('Reconstructed'); axes[1].axis('off')
    if title: fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_reconstruction_grid(originals, reconstructions, nrow=4, title=None, save_path=None):
    n = min(len(originals), nrow * 2)
    fig, axes = plt.subplots(2, nrow, figsize=(3 * nrow, 6))
    for i in range(nrow):
        if i < n:
            axes[0, i].imshow(tensor_to_image(originals[i]))
            axes[1, i].imshow(tensor_to_image(reconstructions[i]))
        axes[0, i].axis('off'); axes[1, i].axis('off')
    axes[0, 0].set_title('Original', fontsize=12)
    axes[1, 0].set_title('Reconstructed', fontsize=12)
    if title: fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_loss_curve(losses, title='Training Loss', ylabel='MSE Loss', save_path=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(losses, linewidth=1.5)
    ax.set_xlabel('Step'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_yscale('log'); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_super_resolution(low_res, reconstructed_hr, ground_truth_hr, title=None, save_path=None):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(tensor_to_image(low_res))
    axes[0].set_title(f'Low-Res ({low_res.shape[-2]}x{low_res.shape[-1]})'); axes[0].axis('off')
    axes[1].imshow(tensor_to_image(reconstructed_hr))
    axes[1].set_title(f'Super-Resolved ({reconstructed_hr.shape[-2]}x{reconstructed_hr.shape[-1]})'); axes[1].axis('off')
    axes[2].imshow(tensor_to_image(ground_truth_hr))
    axes[2].set_title(f'Ground Truth ({ground_truth_hr.shape[-2]}x{ground_truth_hr.shape[-1]})'); axes[2].axis('off')
    if title: fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_architecture_comparison(results_dict, metrics=('psnr', 'ssim', 'lpips'), save_path=None):
    architectures = list(results_dict.keys())
    n_metrics = len(metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    if n_metrics == 1: axes = [axes]
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
    for i, metric in enumerate(metrics):
        values = [results_dict[arch].get(metric, 0) for arch in architectures]
        bars = axes[i].bar(architectures, values, color=colors[:len(architectures)])
        axes[i].set_title(metric.upper(), fontsize=12); axes[i].set_ylabel(metric.upper())
        for bar, val in zip(bars, values):
            axes[i].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                        f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_classification_comparison(results_dict, save_path=None):
    methods = list(results_dict.keys())
    accuracies = list(results_dict.values())
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
    bars = ax.bar(methods, [a * 100 for a in accuracies], color=colors[:len(methods)])
    for bar, acc in zip(bars, accuracies):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=11)
    ax.set_ylabel('Accuracy (%)', fontsize=12)
    ax.set_title('Downstream Classification Comparison', fontsize=14)
    ax.set_ylim(0, 100); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

---
---
# Notebook 01: Coordinate Network Baseline

**Purpose:** Demonstrate understanding of INRs by training coordinate-based networks on single images.
- Train SIREN, ReLU MLP, and Fourier Feature Network
- Compare reconstruction quality and spectral bias

In [ ]:
set_seed(42)

## 1.1 Load a Single Image from CIFAR-10

In [ ]:
cifar = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())

image, label = cifar[0]
print(f'Image shape: {image.shape}, Label: {cifar.classes[label]}')
print(f'Pixel range: [{image.min():.3f}, {image.max():.3f}]')

plt.figure(figsize=(3, 3))
plt.imshow(tensor_to_image(image))
plt.title(f'Target Image: {cifar.classes[label]}')
plt.axis('off')
plt.show()

In [ ]:
coords, pixels = image_to_coord_target(image)
print(f'Coordinates shape: {coords.shape}')  # (1024, 2)
print(f'Pixel targets shape: {pixels.shape}')  # (1024, 3)
print(f'Coordinate range: [{coords.min():.2f}, {coords.max():.2f}]')

## 1.2 Train SIREN on a Single Image

In [ ]:
siren = SIREN(
    in_features=2, hidden_features=256, hidden_layers=3,
    out_features=3, outermost_linear=True,
    first_omega_0=30.0, hidden_omega_0=30.0
)
print(f'SIREN parameters: {siren.get_num_params():,}')

siren_trained, siren_losses = train_single_image_inr(
    siren, coords, pixels, num_steps=2000, lr=1e-4, device=device
)

In [ ]:
siren_recon = reconstruct_image(siren_trained, H=32, W=32, device=device).cpu()
plot_reconstruction(image, siren_recon, title='SIREN Reconstruction',
                    save_path='outputs/figures/reconstructions/siren_single_image.png')

psnr_val = compute_psnr(siren_recon.unsqueeze(0), image.unsqueeze(0)).item()
ssim_val = compute_ssim(siren_recon.unsqueeze(0), image.unsqueeze(0)).item()
print(f'SIREN - PSNR: {psnr_val:.2f} dB, SSIM: {ssim_val:.4f}')

## 1.3 Train ReLU MLP Baseline

In [ ]:
relu_mlp = ReLUCoordinateMLP(in_features=2, hidden_features=256, hidden_layers=3, out_features=3)
print(f'ReLU MLP parameters: {sum(p.numel() for p in relu_mlp.parameters()):,}')

relu_trained, relu_losses = train_single_image_inr(
    relu_mlp, coords, pixels, num_steps=2000, lr=1e-4, device=device
)

In [ ]:
relu_recon = reconstruct_image(relu_trained, H=32, W=32, device=device).cpu()
plot_reconstruction(image, relu_recon, title='ReLU MLP Reconstruction',
                    save_path='outputs/figures/reconstructions/relu_single_image.png')

psnr_relu = compute_psnr(relu_recon.unsqueeze(0), image.unsqueeze(0)).item()
ssim_relu = compute_ssim(relu_recon.unsqueeze(0), image.unsqueeze(0)).item()
print(f'ReLU MLP - PSNR: {psnr_relu:.2f} dB, SSIM: {ssim_relu:.4f}')

## 1.4 Train Fourier Feature Network

In [ ]:
fourier_net = FourierFeatureNetwork(
    in_features=2, mapping_size=256, scale=10.0,
    hidden_features=256, hidden_layers=3, out_features=3
)
print(f'Fourier Feature Network parameters: {sum(p.numel() for p in fourier_net.parameters()):,}')

fourier_trained, fourier_losses = train_single_image_inr(
    fourier_net, coords, pixels, num_steps=2000, lr=1e-4, device=device
)

In [ ]:
fourier_recon = reconstruct_image(fourier_trained, H=32, W=32, device=device).cpu()
plot_reconstruction(image, fourier_recon, title='Fourier Feature Network Reconstruction',
                    save_path='outputs/figures/reconstructions/fourier_single_image.png')

psnr_fourier = compute_psnr(fourier_recon.unsqueeze(0), image.unsqueeze(0)).item()
ssim_fourier = compute_ssim(fourier_recon.unsqueeze(0), image.unsqueeze(0)).item()
print(f'Fourier - PSNR: {psnr_fourier:.2f} dB, SSIM: {ssim_fourier:.4f}')

## 1.5 Architecture Comparison

In [ ]:
# Loss curves comparison
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(siren_losses, label='SIREN', linewidth=1.5)
ax.plot(relu_losses, label='ReLU MLP', linewidth=1.5)
ax.plot(fourier_losses, label='Fourier Features', linewidth=1.5)
ax.set_xlabel('Training Step'); ax.set_ylabel('MSE Loss')
ax.set_title('Training Loss Comparison: Single Image INR')
ax.set_yscale('log'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/comparisons/single_image_loss_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side reconstruction comparison
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
axes[0].imshow(tensor_to_image(image)); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(tensor_to_image(siren_recon)); axes[1].set_title(f'SIREN\nPSNR: {psnr_val:.1f} dB'); axes[1].axis('off')
axes[2].imshow(tensor_to_image(relu_recon)); axes[2].set_title(f'ReLU MLP\nPSNR: {psnr_relu:.1f} dB'); axes[2].axis('off')
axes[3].imshow(tensor_to_image(fourier_recon)); axes[3].set_title(f'Fourier Features\nPSNR: {psnr_fourier:.1f} dB'); axes[3].axis('off')
plt.suptitle('Architecture Comparison: Single Image INR', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/figures/comparisons/single_image_architecture_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Metrics summary table
results_nb01 = pd.DataFrame({
    'Architecture': ['SIREN', 'ReLU MLP', 'Fourier Features'],
    'PSNR (dB)': [psnr_val, psnr_relu, psnr_fourier],
    'SSIM': [ssim_val, ssim_relu, ssim_fourier],
    'Final Loss': [siren_losses[-1], relu_losses[-1], fourier_losses[-1]],
})
print('\n=== Single Image INR Results ===')
print(results_nb01.to_string(index=False))
results_nb01.to_csv('outputs/logs/single_image_inr_results.csv', index=False)

## 1.6 Higher Resolution Reconstruction (Super-Resolution Preview)

Since INRs represent continuous functions, we can query them at any resolution.

In [ ]:
siren_64 = reconstruct_image(siren_trained, H=64, W=64, device=device).cpu()
siren_128 = reconstruct_image(siren_trained, H=128, W=128, device=device).cpu()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(tensor_to_image(siren_recon)); axes[0].set_title('32x32 (training res)'); axes[0].axis('off')
axes[1].imshow(tensor_to_image(siren_64)); axes[1].set_title('64x64 (2x super-res)'); axes[1].axis('off')
axes[2].imshow(tensor_to_image(siren_128)); axes[2].set_title('128x128 (4x super-res)'); axes[2].axis('off')
plt.suptitle('SIREN: Continuous Resolution Querying', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/figures/reconstructions/siren_multiresolution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Notebook 01 complete! Key takeaway: SIREN captures high-frequency details better than ReLU.')

---
---
# Notebook 02: Hypernetwork Training on CIFAR-10

**Purpose:** Train a hypernetwork H_phi(I) -> theta that generates SIREN weights for image reconstruction.

In [ ]:
set_seed(42)

## 2.1 Load CIFAR-10 Data

In [ ]:
train_loader, test_loader = get_cifar10_loaders(root='./data', batch_size=64, num_workers=2)
print(f'Training batches: {len(train_loader)}')
print(f'Test batches: {len(test_loader)}')

sample_images, sample_labels = next(iter(train_loader))
print(f'Batch shape: {sample_images.shape}')

## 2.2 Build Hypernetwork

In [ ]:
LATENT_DIM = 256
SIREN_HIDDEN = 256
SIREN_LAYERS = 3
OMEGA_0 = 30.0

hypernet = HyperNetwork(
    img_shape=(3, 32, 32), latent_dim=LATENT_DIM,
    siren_in_features=2, siren_hidden_features=SIREN_HIDDEN,
    siren_hidden_layers=SIREN_LAYERS, siren_out_features=3
).to(device)

total_params = sum(p.numel() for p in hypernet.parameters())
print(f'Hypernetwork parameters: {total_params:,}')
target_params = sum(fi * fo + fo for fi, fo in hypernet.layer_shapes)
print(f'Target SIREN per-image parameters: {target_params:,}')

## 2.3 Train Hypernetwork

In [ ]:
NUM_EPOCHS = 50
LR = 1e-4

optimizer = torch.optim.Adam(hypernet.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
coords_32 = get_coordinate_grid(32, 32).to(device)

train_losses_nb02 = []
test_losses_nb02 = []
best_test_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    hypernet.train()
    epoch_loss, num_batches = 0.0, 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for images, _ in pbar:
        images = images.to(device)
        B, C, H, W = images.shape
        target = images.permute(0, 2, 3, 1).reshape(B, H * W, C)
        weights, biases, latent = hypernet(images)
        predicted = apply_generated_weights(coords_32, weights, biases, OMEGA_0)
        loss = F.mse_loss(predicted, target)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(hypernet.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        num_batches += 1
        pbar.set_postfix(loss=f'{loss.item():.6f}', psnr=f'{-10*np.log10(loss.item()+1e-10):.1f}')

    scheduler.step()
    avg_train = epoch_loss / num_batches
    train_losses_nb02.append(avg_train)

    # Evaluate on test set
    hypernet.eval()
    test_loss, test_batches = 0.0, 0
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            B = images.shape[0]
            target = images.permute(0, 2, 3, 1).reshape(B, 32 * 32, 3)
            weights, biases, _ = hypernet(images)
            predicted = apply_generated_weights(coords_32, weights, biases, OMEGA_0)
            test_loss += F.mse_loss(predicted, target).item()
            test_batches += 1
    avg_test = test_loss / test_batches
    test_losses_nb02.append(avg_test)
    print(f'  Epoch {epoch+1}: train_loss={avg_train:.6f}, test_loss={avg_test:.6f}, '
          f'PSNR={-10*np.log10(avg_test+1e-10):.2f} dB')
    if avg_test < best_test_loss:
        best_test_loss = avg_test
        torch.save(hypernet.state_dict(), 'outputs/models/checkpoints/hypernet_cifar_best.pt')
        print(f'  -> Saved best model!')

In [ ]:
torch.save(hypernet.state_dict(), 'outputs/models/checkpoints/hypernet_cifar_final.pt')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses_nb02, label='Train', linewidth=1.5)
ax1.plot(test_losses_nb02, label='Test', linewidth=1.5)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE Loss'); ax1.set_title('Training Loss')
ax1.set_yscale('log'); ax1.legend(); ax1.grid(True, alpha=0.3)

psnr_train = [-10 * np.log10(l + 1e-10) for l in train_losses_nb02]
psnr_test = [-10 * np.log10(l + 1e-10) for l in test_losses_nb02]
ax2.plot(psnr_train, label='Train', linewidth=1.5)
ax2.plot(psnr_test, label='Test', linewidth=1.5)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('PSNR (dB)'); ax2.set_title('Reconstruction Quality')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/comparisons/cifar_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Visualize Reconstructions

In [ ]:
hypernet.load_state_dict(torch.load('outputs/models/checkpoints/hypernet_cifar_best.pt', map_location=device))
hypernet.eval()

test_images, test_labels = next(iter(test_loader))
test_images = test_images[:8].to(device)

with torch.no_grad():
    weights, biases, latents = hypernet(test_images)
    recon_images = reconstruct_from_hypernetwork(weights, biases, 32, 32, omega_0=OMEGA_0, device=device)

plot_reconstruction_grid(
    test_images.cpu(), recon_images.cpu(), nrow=8,
    title='CIFAR-10 Hypernetwork Reconstructions',
    save_path='outputs/figures/reconstructions/cifar_hypernet_grid.png'
)

## 2.5 Quantitative Evaluation on Test Set

In [ ]:
lpips_metric = LPIPSMetric(net='alex', device=device)

all_psnr, all_ssim, all_lpips = [], [], []
hypernet.eval()
with torch.no_grad():
    for images, _ in tqdm(test_loader, desc='Evaluating'):
        images = images.to(device)
        weights, biases, _ = hypernet(images)
        recon = reconstruct_from_hypernetwork(weights, biases, 32, 32, omega_0=OMEGA_0, device=device)
        metrics = compute_all_metrics(recon, images, lpips_metric)
        all_psnr.append(metrics['psnr'])
        all_ssim.append(metrics['ssim'])
        if 'lpips' in metrics:
            all_lpips.append(metrics['lpips'])

print(f'\n=== CIFAR-10 Test Set Results ===')
print(f'PSNR:  {np.mean(all_psnr):.2f} +/- {np.std(all_psnr):.2f} dB')
print(f'SSIM:  {np.mean(all_ssim):.4f} +/- {np.std(all_ssim):.4f}')
if all_lpips:
    print(f'LPIPS: {np.mean(all_lpips):.4f} +/- {np.std(all_lpips):.4f}')

## 2.6 Extract and Save Features for Downstream Classification

In [ ]:
def extract_features(hypernet, data_loader, device):
    """Extract latent features and generated weights from hypernetwork."""
    hypernet.eval()
    all_latents, all_weights_flat, all_labels, all_pixels = [], [], [], []
    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc='Extracting features'):
            images = images.to(device)
            weights, biases, latent = hypernet(images)
            weight_flat = torch.cat(
                [w.reshape(w.shape[0], -1) for w in weights] +
                [b.reshape(b.shape[0], -1) for b in biases], dim=1
            )
            all_latents.append(latent.cpu())
            all_weights_flat.append(weight_flat.cpu())
            all_labels.append(labels)
            all_pixels.append(images.cpu().reshape(images.shape[0], -1))
    return {
        'latents': torch.cat(all_latents, dim=0),
        'weights': torch.cat(all_weights_flat, dim=0),
        'labels': torch.cat(all_labels, dim=0),
        'pixels': torch.cat(all_pixels, dim=0),
    }

print('Extracting training features...')
train_features = extract_features(hypernet, train_loader, device)
print('Extracting test features...')
test_features = extract_features(hypernet, test_loader, device)

torch.save(train_features, 'outputs/models/cifar_train_features.pt')
torch.save(test_features, 'outputs/models/cifar_test_features.pt')

print(f'\nFeature shapes:')
print(f'  Latent: {train_features["latents"].shape}')
print(f'  Weights: {train_features["weights"].shape}')
print(f'  Pixels: {train_features["pixels"].shape}')
print(f'  Labels: {train_features["labels"].shape}')
print('\nNotebook 02 complete! Model and features saved.')

---
---
# Notebook 03: Downstream Classification

**Purpose:** Investigate whether INR representations capture meaningful semantic information.
- Compare: Raw pixels vs Latent z vs Generated INR weights
- Using Linear and MLP classifiers for CIFAR-10

In [ ]:
set_seed(42)

## 3.1 Load Extracted Features

In [ ]:
# Use features already in memory from Notebook 02 (or load from disk)
try:
    _ = train_features['latents']
    print('Using features from memory (Notebook 02).')
except NameError:
    train_features = torch.load('outputs/models/cifar_train_features.pt', map_location='cpu')
    test_features = torch.load('outputs/models/cifar_test_features.pt', map_location='cpu')
    print('Loaded features from disk.')

print('Feature dimensions:')
print(f'  Pixels:  train={train_features["pixels"].shape}, test={test_features["pixels"].shape}')
print(f'  Latent:  train={train_features["latents"].shape}, test={test_features["latents"].shape}')
print(f'  Weights: train={train_features["weights"].shape}, test={test_features["weights"].shape}')
print(f'  Labels:  train={train_features["labels"].shape}, test={test_features["labels"].shape}')

## 3.2 Define Classifiers

In [ ]:
class LinearClassifier(nn.Module):
    def __init__(self, in_features, num_classes=10):
        super().__init__()
        self.fc = nn.Linear(in_features, num_classes)
    def forward(self, x):
        return self.fc(x)


class MLPClassifier(nn.Module):
    def __init__(self, in_features, num_classes=10, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, num_classes),
        )
    def forward(self, x):
        return self.net(x)


def train_classifier(model, train_loader, test_loader, epochs=30, lr=1e-3, device='cuda'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    train_accs, test_accs = [], []
    for epoch in range(epochs):
        model.train()
        correct, total = 0, 0
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            logits = model(features)
            loss = F.cross_entropy(logits, labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
        scheduler.step()
        train_accs.append(correct / total)
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for features, labels in test_loader:
                features, labels = features.to(device), labels.to(device)
                logits = model(features)
                correct += (logits.argmax(1) == labels).sum().item()
                total += labels.size(0)
        test_accs.append(correct / total)
    return test_accs[-1], train_accs, test_accs

## 3.3 Prepare DataLoaders

In [ ]:
def make_loaders(train_x, train_y, test_x, test_y, batch_size=256):
    mean = train_x.mean(0, keepdim=True)
    std = train_x.std(0, keepdim=True) + 1e-8
    train_x = (train_x - mean) / std
    test_x = (test_x - mean) / std
    train_ds = TensorDataset(train_x, train_y)
    test_ds = TensorDataset(test_x, test_y)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            DataLoader(test_ds, batch_size=batch_size, shuffle=False))

pixel_train_loader, pixel_test_loader = make_loaders(
    train_features['pixels'], train_features['labels'],
    test_features['pixels'], test_features['labels'])

latent_train_loader, latent_test_loader = make_loaders(
    train_features['latents'], train_features['labels'],
    test_features['latents'], test_features['labels'])

weight_dim = train_features['weights'].shape[1]
print(f'Weight feature dimension: {weight_dim}')

if weight_dim > 2000:
    print('Applying PCA to reduce weight dimensionality...')
    pca = PCA(n_components=512)
    weights_train_pca = torch.tensor(pca.fit_transform(train_features['weights'].numpy()), dtype=torch.float32)
    weights_test_pca = torch.tensor(pca.transform(test_features['weights'].numpy()), dtype=torch.float32)
    print(f'Explained variance ratio (sum): {pca.explained_variance_ratio_.sum():.4f}')
    weight_train_loader, weight_test_loader = make_loaders(
        weights_train_pca, train_features['labels'],
        weights_test_pca, test_features['labels'])
    weight_input_dim = 512
else:
    weight_train_loader, weight_test_loader = make_loaders(
        train_features['weights'], train_features['labels'],
        test_features['weights'], test_features['labels'])
    weight_input_dim = weight_dim

## 3.4 Train and Evaluate All Classifiers

In [ ]:
EPOCHS = 30
results_cls = {}

print('\n=== LINEAR CLASSIFIERS ===')
set_seed(42)
acc, _, _ = train_classifier(LinearClassifier(3072), pixel_train_loader, pixel_test_loader, epochs=EPOCHS, device=device)
results_cls['Pixels (Linear)'] = acc; print(f'Pixels (Linear): {acc*100:.1f}%')

set_seed(42)
acc, _, _ = train_classifier(LinearClassifier(train_features['latents'].shape[1]), latent_train_loader, latent_test_loader, epochs=EPOCHS, device=device)
results_cls['Latent z (Linear)'] = acc; print(f'Latent z (Linear): {acc*100:.1f}%')

set_seed(42)
acc, _, _ = train_classifier(LinearClassifier(weight_input_dim), weight_train_loader, weight_test_loader, epochs=EPOCHS, device=device)
results_cls['INR Weights (Linear)'] = acc; print(f'INR Weights (Linear): {acc*100:.1f}%')

print('\n=== MLP CLASSIFIERS ===')
set_seed(42)
acc, pixel_train_hist, pixel_test_hist = train_classifier(MLPClassifier(3072), pixel_train_loader, pixel_test_loader, epochs=EPOCHS, device=device)
results_cls['Pixels (MLP)'] = acc; print(f'Pixels (MLP): {acc*100:.1f}%')

set_seed(42)
acc, latent_train_hist, latent_test_hist = train_classifier(MLPClassifier(train_features['latents'].shape[1]), latent_train_loader, latent_test_loader, epochs=EPOCHS, device=device)
results_cls['Latent z (MLP)'] = acc; print(f'Latent z (MLP): {acc*100:.1f}%')

set_seed(42)
acc, weight_train_hist, weight_test_hist = train_classifier(MLPClassifier(weight_input_dim), weight_train_loader, weight_test_loader, epochs=EPOCHS, device=device)
results_cls['INR Weights (MLP)'] = acc; print(f'INR Weights (MLP): {acc*100:.1f}%')

## 3.5 Results Visualization

In [ ]:
results_df = pd.DataFrame([{'Method': k, 'Accuracy (%)': f'{v*100:.1f}'} for k, v in results_cls.items()])
print('\n=== Classification Results ===')
print(results_df.to_string(index=False))
results_df.to_csv('outputs/logs/classification_results.csv', index=False)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800']

linear_methods = ['Pixels (Linear)', 'Latent z (Linear)', 'INR Weights (Linear)']
linear_accs = [results_cls[m] * 100 for m in linear_methods]
bars1 = ax1.bar(['Pixels', 'Latent z', 'INR Weights'], linear_accs, color=colors)
for bar, acc in zip(bars1, linear_accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{acc:.1f}%', ha='center', fontsize=11)
ax1.set_ylabel('Accuracy (%)'); ax1.set_title('Linear Classifier'); ax1.set_ylim(0, 100); ax1.grid(axis='y', alpha=0.3)

mlp_methods = ['Pixels (MLP)', 'Latent z (MLP)', 'INR Weights (MLP)']
mlp_accs = [results_cls[m] * 100 for m in mlp_methods]
bars2 = ax2.bar(['Pixels', 'Latent z', 'INR Weights'], mlp_accs, color=colors)
for bar, acc in zip(bars2, mlp_accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{acc:.1f}%', ha='center', fontsize=11)
ax2.set_ylabel('Accuracy (%)'); ax2.set_title('MLP Classifier'); ax2.set_ylim(0, 100); ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Downstream CIFAR-10 Classification: Feature Representation Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/figures/comparisons/classification_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pixel_test_hist, label='Pixels', linewidth=1.5)
ax.plot(latent_test_hist, label='Latent z', linewidth=1.5)
ax.plot(weight_test_hist, label='INR Weights', linewidth=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Test Accuracy')
ax.set_title('MLP Classifier: Test Accuracy Over Training')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/comparisons/classification_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNotebook 03 complete! Key insight: Latent representations capture semantic information useful for classification.')

---
---
# Notebook 04: CelebA Meta-Learning Experiments

**Purpose:** Compare INR architectures (SIREN, Fourier, ReLU) on CelebA 64x64 faces.

**Note:** CelebA must be downloaded. If automatic download fails, upload manually to `./data/celeba/`.
You can download it from: https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html

In [ ]:
set_seed(42)

## 4.1 Load CelebA Data

In [ ]:
IMAGE_SIZE = 64
BATCH_SIZE_CELEBA = 32
MAX_TRAIN = 20000

train_loader_celeba, val_loader_celeba, test_loader_celeba = get_celeba_loaders(
    root='./data', image_size=IMAGE_SIZE, batch_size=BATCH_SIZE_CELEBA,
    num_workers=2, max_train=MAX_TRAIN
)
print(f'Training batches: {len(train_loader_celeba)}')
print(f'Validation batches: {len(val_loader_celeba)}')
print(f'Test batches: {len(test_loader_celeba)}')

sample_images_celeba, _ = next(iter(train_loader_celeba))
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i in range(8):
    axes[i].imshow(tensor_to_image(sample_images_celeba[i])); axes[i].axis('off')
plt.suptitle('CelebA Samples (64x64)', fontsize=14)
plt.tight_layout(); plt.show()

## 4.2 Architecture Comparison Setup

In [ ]:
LATENT_DIM_CELEBA = 512
SIREN_HIDDEN_CELEBA = 256
SIREN_LAYERS_CELEBA = 3
NUM_EPOCHS_CELEBA = 30
LR_CELEBA = 1e-4

def train_celeba_hypernet(hypernet, train_loader, val_loader, omega_0, num_epochs,
                          lr, device, save_name):
    optimizer = torch.optim.Adam(hypernet.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    coords = get_coordinate_grid(IMAGE_SIZE, IMAGE_SIZE).to(device)
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    for epoch in range(num_epochs):
        hypernet.train()
        epoch_loss, num_batches = 0.0, 0
        pbar = tqdm(train_loader, desc=f'[{save_name}] Epoch {epoch+1}/{num_epochs}', leave=False)
        for images, _ in pbar:
            images = images.to(device)
            B, C, H, W = images.shape
            target = images.permute(0, 2, 3, 1).reshape(B, H * W, C)
            weights, biases, _ = hypernet(images)
            predicted = apply_generated_weights(coords, weights, biases, omega_0)
            loss = F.mse_loss(predicted, target)
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(hypernet.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item(); num_batches += 1
            pbar.set_postfix(loss=f'{loss.item():.6f}')
        scheduler.step()
        train_losses.append(epoch_loss / num_batches)
        hypernet.eval()
        val_loss, val_batches = 0.0, 0
        with torch.no_grad():
            for images, _ in val_loader:
                images = images.to(device)
                B = images.shape[0]
                target = images.permute(0, 2, 3, 1).reshape(B, -1, 3)
                weights, biases, _ = hypernet(images)
                predicted = apply_generated_weights(coords, weights, biases, omega_0)
                val_loss += F.mse_loss(predicted, target).item(); val_batches += 1
                if val_batches >= 50: break
        avg_val = val_loss / val_batches
        val_losses.append(avg_val)
        psnr = -10 * np.log10(avg_val + 1e-10)
        print(f'  [{save_name}] Epoch {epoch+1}: train={train_losses[-1]:.6f}, val={avg_val:.6f}, PSNR={psnr:.2f}')
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(hypernet.state_dict(), f'outputs/models/checkpoints/hypernet_celeba_{save_name}.pt')
    return {'train_loss': train_losses, 'val_loss': val_losses}

## 4.3 Train SIREN-based Hypernetwork

In [ ]:
set_seed(42)
hypernet_siren = HyperNetwork(
    img_shape=(3, IMAGE_SIZE, IMAGE_SIZE), latent_dim=LATENT_DIM_CELEBA,
    siren_hidden_features=SIREN_HIDDEN_CELEBA, siren_hidden_layers=SIREN_LAYERS_CELEBA,
).to(device)
print(f'SIREN Hypernetwork params: {sum(p.numel() for p in hypernet_siren.parameters()):,}')

history_siren = train_celeba_hypernet(
    hypernet_siren, train_loader_celeba, val_loader_celeba,
    omega_0=30.0, num_epochs=NUM_EPOCHS_CELEBA, lr=LR_CELEBA,
    device=device, save_name='siren'
)

## 4.4 Train Fourier Feature Hypernetwork

In [ ]:
set_seed(42)
hypernet_fourier = HyperNetwork(
    img_shape=(3, IMAGE_SIZE, IMAGE_SIZE), latent_dim=LATENT_DIM_CELEBA,
    siren_hidden_features=SIREN_HIDDEN_CELEBA, siren_hidden_layers=SIREN_LAYERS_CELEBA,
).to(device)

FOURIER_OMEGA = 10.0

history_fourier = train_celeba_hypernet(
    hypernet_fourier, train_loader_celeba, val_loader_celeba,
    omega_0=FOURIER_OMEGA, num_epochs=NUM_EPOCHS_CELEBA, lr=LR_CELEBA,
    device=device, save_name='fourier'
)

## 4.5 Train ReLU Hypernetwork

In [ ]:
def apply_generated_weights_relu(coords, weights, biases):
    """Forward pass with ReLU activations instead of sine."""
    if coords.dim() == 2:
        coords = coords.unsqueeze(0).expand(weights[0].shape[0], -1, -1)
    x = coords
    num_layers = len(weights)
    for i in range(num_layers):
        x = torch.bmm(x, weights[i].transpose(1, 2)) + biases[i].unsqueeze(1)
        if i < num_layers - 1:
            x = F.relu(x)
    return x


set_seed(42)
hypernet_relu = HyperNetwork(
    img_shape=(3, IMAGE_SIZE, IMAGE_SIZE), latent_dim=LATENT_DIM_CELEBA,
    siren_hidden_features=SIREN_HIDDEN_CELEBA, siren_hidden_layers=SIREN_LAYERS_CELEBA,
).to(device)

optimizer_relu = torch.optim.Adam(hypernet_relu.parameters(), lr=LR_CELEBA)
scheduler_relu = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_relu, T_max=NUM_EPOCHS_CELEBA)
coords_celeba = get_coordinate_grid(IMAGE_SIZE, IMAGE_SIZE).to(device)

relu_train_losses, relu_val_losses = [], []
best_val_relu = float('inf')

for epoch in range(NUM_EPOCHS_CELEBA):
    hypernet_relu.train()
    epoch_loss, num_b = 0.0, 0
    pbar = tqdm(train_loader_celeba, desc=f'[ReLU] Epoch {epoch+1}/{NUM_EPOCHS_CELEBA}', leave=False)
    for images, _ in pbar:
        images = images.to(device)
        B, C, H, W = images.shape
        target = images.permute(0, 2, 3, 1).reshape(B, H*W, C)
        weights, biases, _ = hypernet_relu(images)
        predicted = apply_generated_weights_relu(coords_celeba, weights, biases)
        loss = F.mse_loss(predicted, target)
        optimizer_relu.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(hypernet_relu.parameters(), 1.0)
        optimizer_relu.step()
        epoch_loss += loss.item(); num_b += 1
        pbar.set_postfix(loss=f'{loss.item():.6f}')
    scheduler_relu.step()
    relu_train_losses.append(epoch_loss / num_b)
    hypernet_relu.eval()
    vl, vb = 0.0, 0
    with torch.no_grad():
        for images, _ in val_loader_celeba:
            images = images.to(device)
            B = images.shape[0]
            target = images.permute(0, 2, 3, 1).reshape(B, -1, 3)
            w, b, _ = hypernet_relu(images)
            pred = apply_generated_weights_relu(coords_celeba, w, b)
            vl += F.mse_loss(pred, target).item(); vb += 1
            if vb >= 50: break
    avg_vl = vl / vb
    relu_val_losses.append(avg_vl)
    print(f'  [ReLU] Epoch {epoch+1}: train={relu_train_losses[-1]:.6f}, val={avg_vl:.6f}, PSNR={-10*np.log10(avg_vl+1e-10):.2f}')
    if avg_vl < best_val_relu:
        best_val_relu = avg_vl
        torch.save(hypernet_relu.state_dict(), 'outputs/models/checkpoints/hypernet_celeba_relu.pt')

history_relu = {'train_loss': relu_train_losses, 'val_loss': relu_val_losses}

## 4.6 Training Curve Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history_siren['train_loss'], label='SIREN', linewidth=1.5)
ax1.plot(history_fourier['train_loss'], label='Fourier', linewidth=1.5)
ax1.plot(history_relu['train_loss'], label='ReLU', linewidth=1.5)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE Loss'); ax1.set_title('Training Loss')
ax1.set_yscale('log'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history_siren['val_loss'], label='SIREN', linewidth=1.5)
ax2.plot(history_fourier['val_loss'], label='Fourier', linewidth=1.5)
ax2.plot(history_relu['val_loss'], label='ReLU', linewidth=1.5)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MSE Loss'); ax2.set_title('Validation Loss')
ax2.set_yscale('log'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('CelebA Architecture Comparison: Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/figures/comparisons/celeba_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 Quantitative Evaluation

In [ ]:
lpips_metric_celeba = LPIPSMetric(net='alex', device=device)

def evaluate_celeba_model(hypernet, test_loader, omega_0, apply_fn, device, max_batches=50):
    hypernet.eval()
    coords = get_coordinate_grid(IMAGE_SIZE, IMAGE_SIZE).to(device)
    all_psnr, all_ssim, all_lpips = [], [], []
    with torch.no_grad():
        for i, (images, _) in enumerate(tqdm(test_loader, desc='Evaluating')):
            if i >= max_batches: break
            images = images.to(device)
            weights, biases, _ = hypernet(images)
            if apply_fn == 'siren':
                pred_flat = apply_generated_weights(coords, weights, biases, omega_0)
            else:
                pred_flat = apply_generated_weights_relu(coords, weights, biases)
            B = images.shape[0]
            recon = pred_flat.clamp(0, 1).reshape(B, IMAGE_SIZE, IMAGE_SIZE, 3).permute(0, 3, 1, 2)
            metrics = compute_all_metrics(recon, images, lpips_metric_celeba)
            all_psnr.append(metrics['psnr']); all_ssim.append(metrics['ssim'])
            if 'lpips' in metrics: all_lpips.append(metrics['lpips'])
    return {'psnr': np.mean(all_psnr), 'ssim': np.mean(all_ssim),
            'lpips': np.mean(all_lpips) if all_lpips else 0.0}

# Load best checkpoints
hypernet_siren.load_state_dict(torch.load('outputs/models/checkpoints/hypernet_celeba_siren.pt', map_location=device))
hypernet_fourier.load_state_dict(torch.load('outputs/models/checkpoints/hypernet_celeba_fourier.pt', map_location=device))
hypernet_relu.load_state_dict(torch.load('outputs/models/checkpoints/hypernet_celeba_relu.pt', map_location=device))

print('\nEvaluating SIREN...')
results_siren = evaluate_celeba_model(hypernet_siren, test_loader_celeba, 30.0, 'siren', device)
print('Evaluating Fourier...')
results_fourier = evaluate_celeba_model(hypernet_fourier, test_loader_celeba, FOURIER_OMEGA, 'siren', device)
print('Evaluating ReLU...')
results_relu_celeba = evaluate_celeba_model(hypernet_relu, test_loader_celeba, 1.0, 'relu', device)

In [ ]:
results_dict_celeba = {'SIREN': results_siren, 'Fourier': results_fourier, 'ReLU': results_relu_celeba}
df_celeba = pd.DataFrame(results_dict_celeba).T
df_celeba.columns = ['PSNR (dB)', 'SSIM', 'LPIPS']
print('\n=== CelebA Architecture Comparison ===')
print(df_celeba.to_string())
df_celeba.to_csv('outputs/logs/celeba_architecture_comparison.csv')

plot_architecture_comparison(
    results_dict_celeba, metrics=('psnr', 'ssim', 'lpips'),
    save_path='outputs/figures/comparisons/celeba_architecture_metrics.png'
)

## 4.8 Visual Reconstruction Comparison

In [ ]:
test_imgs_celeba, _ = next(iter(test_loader_celeba))
test_imgs_celeba = test_imgs_celeba[:6].to(device)
coords_vis = get_coordinate_grid(IMAGE_SIZE, IMAGE_SIZE).to(device)

with torch.no_grad():
    w, b, _ = hypernet_siren(test_imgs_celeba)
    recon_siren_vis = reconstruct_from_hypernetwork(w, b, IMAGE_SIZE, IMAGE_SIZE, 30.0, device)
    w, b, _ = hypernet_fourier(test_imgs_celeba)
    recon_fourier_vis = reconstruct_from_hypernetwork(w, b, IMAGE_SIZE, IMAGE_SIZE, FOURIER_OMEGA, device)
    w, b, _ = hypernet_relu(test_imgs_celeba)
    pred_relu_vis = apply_generated_weights_relu(coords_vis, w, b)
    recon_relu_vis = pred_relu_vis.clamp(0, 1).reshape(-1, IMAGE_SIZE, IMAGE_SIZE, 3).permute(0, 3, 1, 2)

n_show = 6
fig, axes = plt.subplots(4, n_show, figsize=(3 * n_show, 12))
row_labels = ['Original', 'SIREN', 'Fourier', 'ReLU']
images_list = [test_imgs_celeba.cpu(), recon_siren_vis.cpu(), recon_fourier_vis.cpu(), recon_relu_vis.cpu()]
for row, (label, imgs) in enumerate(zip(row_labels, images_list)):
    for col in range(n_show):
        axes[row, col].imshow(tensor_to_image(imgs[col])); axes[row, col].axis('off')
    axes[row, 0].set_ylabel(label, fontsize=12, rotation=0, labelpad=60, va='center')

plt.suptitle('CelebA Reconstruction: Architecture Comparison', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('outputs/figures/comparisons/celeba_recon_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNotebook 04 complete! Architecture comparison done.')

---
---
# Notebook 05: Super-Resolution via INR

**Purpose:** Demonstrate super-resolution by querying coordinate networks at higher resolutions.
- Train on 32x32, query at 64x64, 128x128, 256x256
- Compare against bicubic upsampling

In [ ]:
set_seed(42)

## 5.1 Load CelebA at High Resolution (for Ground Truth)

In [ ]:
HR_SIZE = 128
LR_SIZE = 32
BATCH_SIZE_SR = 32
MAX_TRAIN_SR = 20000

train_loader_hr, val_loader_hr, test_loader_hr = get_celeba_loaders(
    root='./data', image_size=HR_SIZE, batch_size=BATCH_SIZE_SR,
    num_workers=2, max_train=MAX_TRAIN_SR
)
print(f'High-res size: {HR_SIZE}x{HR_SIZE}')
print(f'Low-res (training) size: {LR_SIZE}x{LR_SIZE}')
print(f'Training batches: {len(train_loader_hr)}')

## 5.2 Train Hypernetwork on Low-Resolution Images

In [ ]:
LATENT_DIM_SR = 512
SIREN_HIDDEN_SR = 256
SIREN_LAYERS_SR = 3
OMEGA_0_SR = 30.0
NUM_EPOCHS_SR = 30
LR_SR = 1e-4

hypernet_sr = HyperNetwork(
    img_shape=(3, LR_SIZE, LR_SIZE), latent_dim=LATENT_DIM_SR,
    siren_hidden_features=SIREN_HIDDEN_SR, siren_hidden_layers=SIREN_LAYERS_SR,
).to(device)
print(f'SR Hypernetwork params: {sum(p.numel() for p in hypernet_sr.parameters()):,}')

optimizer_sr = torch.optim.Adam(hypernet_sr.parameters(), lr=LR_SR)
scheduler_sr = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_sr, T_max=NUM_EPOCHS_SR)
coords_lr = get_coordinate_grid(LR_SIZE, LR_SIZE).to(device)

train_losses_sr = []
best_val_loss_sr = float('inf')

for epoch in range(NUM_EPOCHS_SR):
    hypernet_sr.train()
    epoch_loss, num_batches = 0.0, 0
    pbar = tqdm(train_loader_hr, desc=f'Epoch {epoch+1}/{NUM_EPOCHS_SR}')
    for images_hr, _ in pbar:
        images_hr = images_hr.to(device)
        images_lr = downsample_images(images_hr, LR_SIZE)
        B, C, H, W = images_lr.shape
        target = images_lr.permute(0, 2, 3, 1).reshape(B, H * W, C)
        weights, biases, _ = hypernet_sr(images_lr)
        predicted = apply_generated_weights(coords_lr, weights, biases, OMEGA_0_SR)
        loss = F.mse_loss(predicted, target)
        optimizer_sr.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(hypernet_sr.parameters(), 1.0)
        optimizer_sr.step()
        epoch_loss += loss.item(); num_batches += 1
        pbar.set_postfix(loss=f'{loss.item():.6f}', psnr=f'{-10*np.log10(loss.item()+1e-10):.1f}')
    scheduler_sr.step()
    avg_loss = epoch_loss / num_batches
    train_losses_sr.append(avg_loss)
    print(f'  Epoch {epoch+1}: loss={avg_loss:.6f}, PSNR={-10*np.log10(avg_loss+1e-10):.2f}')
    torch.save(hypernet_sr.state_dict(), 'outputs/models/checkpoints/hypernet_sr_latest.pt')
    if avg_loss < best_val_loss_sr:
        best_val_loss_sr = avg_loss
        torch.save(hypernet_sr.state_dict(), 'outputs/models/checkpoints/hypernet_sr_best.pt')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses_sr, linewidth=1.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Super-Resolution Hypernetwork Training')
ax.set_yscale('log'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/super_resolution/sr_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.3 Super-Resolution: Query at Multiple Resolutions

In [ ]:
hypernet_sr.load_state_dict(torch.load('outputs/models/checkpoints/hypernet_sr_best.pt', map_location=device))
hypernet_sr.eval()

test_images_hr_batch, _ = next(iter(test_loader_hr))
test_images_hr_batch = test_images_hr_batch[:8].to(device)
test_images_lr_batch = downsample_images(test_images_hr_batch, LR_SIZE)

eval_resolutions = [32, 64, 128, 256]
sr_results = {}
with torch.no_grad():
    weights, biases, _ = hypernet_sr(test_images_lr_batch)
    for res in eval_resolutions:
        coords_eval = get_coordinate_grid(res, res).to(device)
        pred = apply_generated_weights(coords_eval, weights, biases, OMEGA_0_SR)
        B = pred.shape[0]
        recon = pred.clamp(0, 1).reshape(B, res, res, 3).permute(0, 3, 1, 2)
        sr_results[res] = recon.cpu()
        print(f'Resolution {res}x{res}: generated')
print('Super-resolution complete!')

## 5.4 Visual Comparison

In [ ]:
n_show = 4
fig, axes = plt.subplots(n_show, len(eval_resolutions) + 1, figsize=(3 * (len(eval_resolutions) + 1), 3 * n_show))
for row in range(n_show):
    axes[row, 0].imshow(tensor_to_image(test_images_lr_batch[row].cpu()))
    if row == 0: axes[row, 0].set_title(f'Input\n{LR_SIZE}x{LR_SIZE}', fontsize=10)
    axes[row, 0].axis('off')
    for col, res in enumerate(eval_resolutions):
        axes[row, col + 1].imshow(tensor_to_image(sr_results[res][row]))
        if row == 0:
            suffix = ' (trained)' if res == LR_SIZE else f' ({res//LR_SIZE}x SR)'
            axes[row, col + 1].set_title(f'{res}x{res}{suffix}', fontsize=10)
        axes[row, col + 1].axis('off')
plt.suptitle('Super-Resolution: INR Queried at Multiple Resolutions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('outputs/figures/super_resolution/sr_multiresolution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Side-by-side: Low-res | Super-Resolved | Ground Truth
for i in range(min(4, n_show)):
    gt_128 = downsample_images(test_images_hr_batch[i:i+1].cpu(), 128).squeeze(0)
    plot_super_resolution(
        low_res=test_images_lr_batch[i].cpu(),
        reconstructed_hr=sr_results[128][i],
        ground_truth_hr=gt_128,
        title=f'Sample {i+1}: {LR_SIZE}x{LR_SIZE} -> 128x128',
        save_path=f'outputs/figures/super_resolution/sr_comparison_{i}.png'
    )

## 5.5 Quantitative Super-Resolution Evaluation

In [ ]:
lpips_metric_sr = LPIPSMetric(net='alex', device=device)
sr_metrics = {}

hypernet_sr.eval()
for target_res in [64, 128]:
    all_psnr, all_ssim, all_lpips = [], [], []
    coords_eval = get_coordinate_grid(target_res, target_res).to(device)
    with torch.no_grad():
        for batch_idx, (images_hr, _) in enumerate(tqdm(test_loader_hr, desc=f'Eval {target_res}x{target_res}')):
            if batch_idx >= 50: break
            images_hr = images_hr.to(device)
            images_lr = downsample_images(images_hr, LR_SIZE)
            gt_target = downsample_images(images_hr, target_res)
            weights, biases, _ = hypernet_sr(images_lr)
            pred = apply_generated_weights(coords_eval, weights, biases, OMEGA_0_SR)
            B = pred.shape[0]
            recon = pred.clamp(0, 1).reshape(B, target_res, target_res, 3).permute(0, 3, 1, 2)
            metrics = compute_all_metrics(recon, gt_target, lpips_metric_sr)
            all_psnr.append(metrics['psnr']); all_ssim.append(metrics['ssim'])
            if 'lpips' in metrics: all_lpips.append(metrics['lpips'])
    sr_metrics[f'{target_res}x{target_res}'] = {
        'psnr': np.mean(all_psnr), 'ssim': np.mean(all_ssim),
        'lpips': np.mean(all_lpips) if all_lpips else 0.0
    }

# Bicubic baseline
for target_res in [64, 128]:
    all_psnr, all_ssim, all_lpips = [], [], []
    with torch.no_grad():
        for batch_idx, (images_hr, _) in enumerate(tqdm(test_loader_hr, desc=f'Bicubic {target_res}')):
            if batch_idx >= 50: break
            images_hr = images_hr.to(device)
            images_lr = downsample_images(images_hr, LR_SIZE)
            gt_target = downsample_images(images_hr, target_res)
            bicubic = F.interpolate(images_lr, size=target_res, mode='bicubic', align_corners=False).clamp(0, 1)
            metrics = compute_all_metrics(bicubic, gt_target, lpips_metric_sr)
            all_psnr.append(metrics['psnr']); all_ssim.append(metrics['ssim'])
            if 'lpips' in metrics: all_lpips.append(metrics['lpips'])
    sr_metrics[f'Bicubic {target_res}x{target_res}'] = {
        'psnr': np.mean(all_psnr), 'ssim': np.mean(all_ssim),
        'lpips': np.mean(all_lpips) if all_lpips else 0.0
    }

In [ ]:
df_sr = pd.DataFrame(sr_metrics).T
df_sr.columns = ['PSNR (dB)', 'SSIM', 'LPIPS']
print('\n=== Super-Resolution Results ===')
print(df_sr.to_string())
df_sr.to_csv('outputs/logs/super_resolution_results.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methods = ['INR 64x64', 'Bicubic 64x64', 'INR 128x128', 'Bicubic 128x128']
metrics_names = ['PSNR (dB)', 'SSIM', 'LPIPS']
bar_colors = ['#2196F3', '#90CAF9', '#FF9800', '#FFE0B2']

for i, metric in enumerate(metrics_names):
    vals = [
        sr_metrics['64x64'][metric.split()[0].lower()],
        sr_metrics['Bicubic 64x64'][metric.split()[0].lower()],
        sr_metrics['128x128'][metric.split()[0].lower()],
        sr_metrics['Bicubic 128x128'][metric.split()[0].lower()],
    ]
    bars = axes[i].bar(methods, vals, color=bar_colors)
    axes[i].set_title(metric)
    axes[i].tick_params(axis='x', rotation=30)
    for bar, v in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Super-Resolution: INR vs Bicubic Upsampling', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/figures/super_resolution/sr_inr_vs_bicubic.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Final comparison: Low-res | Bicubic | INR | Ground Truth
test_imgs_final, _ = next(iter(test_loader_hr))
test_imgs_final = test_imgs_final[:4].to(device)
test_imgs_lr_final = downsample_images(test_imgs_final, LR_SIZE)

with torch.no_grad():
    weights, biases, _ = hypernet_sr(test_imgs_lr_final)
    coords_128 = get_coordinate_grid(128, 128).to(device)
    pred = apply_generated_weights(coords_128, weights, biases, OMEGA_0_SR)
    inr_128 = pred.clamp(0, 1).reshape(-1, 128, 128, 3).permute(0, 3, 1, 2).cpu()

bicubic_128 = F.interpolate(test_imgs_lr_final.cpu(), size=128, mode='bicubic', align_corners=False).clamp(0, 1)
gt_128 = downsample_images(test_imgs_final.cpu(), 128)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
col_labels = [f'Low-Res ({LR_SIZE}x{LR_SIZE})', 'Bicubic (128x128)', 'INR (128x128)', 'Ground Truth (128x128)']
for row in range(4):
    lr_display = F.interpolate(test_imgs_lr_final[row:row+1].cpu(), size=128, mode='nearest').squeeze(0)
    axes[row, 0].imshow(tensor_to_image(lr_display))
    axes[row, 1].imshow(tensor_to_image(bicubic_128[row]))
    axes[row, 2].imshow(tensor_to_image(inr_128[row]))
    axes[row, 3].imshow(tensor_to_image(gt_128[row]))
    for col in range(4):
        axes[row, col].axis('off')
        if row == 0: axes[row, col].set_title(col_labels[col], fontsize=11)

plt.suptitle(f'Super-Resolution Comparison: {LR_SIZE}x{LR_SIZE} -> 128x128', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('outputs/figures/super_resolution/sr_final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNotebook 05 complete! Super-resolution results saved.')
print('All 5 experiments done - ready for report writing!')